# BLM AIM plots: ALS tiles with measured cover

Shows 3D lidar tiles (USGS 3DEP, streamed from AWS) around BLM AIM/LMF field plots and labels each tile with the **cover % measured on the ground** at that plot.

Input is the matched plot table written by `data_cleaning/blm_aim/blm_aim_lmf_summary.ipynb` (`datasets/BLM_AIM/BLM_AIM_plots.gpkg`), which carries the AWS product-matching columns `product_name_AWS`, `collection_year_AWS` and `year_diff_AWS`. Tile loading, caching and coloring are the same as in [vegbank_class_quiz.ipynb](vegbank_class_quiz.ipynb) (they share the tile cache).

Cover values are percent of line-point hits, so layers overlap and can sum to more than 100.

## Configuration

In [ ]:
GPKG = '../../datasets/BLM_AIM/BLM_AIM_plots.gpkg'
SORT_BY = 'AH_ShrubCover'   # cover column to sweep: one row of tiles per quantile bin of this variable (see COVER_COLS)
N_BINS = 5                  # number of quantile bins (rows) along SORT_BY, low to high
N_PER_BIN = 3               # tiles per bin
SPLIT = None                # None = all plots, 'test' or 'train' = only that side of the test_split column
STATE = None                # e.g. 'NV' to restrict to one state, None for all
RADIUS = 50                 # tile radius in meters
SEED = 0                    # int for reproducible draws, None for different ones every run
MAX_POINTS = 15000          # points per tile are randomly thinned to this many for plotting
COLOR_BY = 'intensity' # 'height', 'intensity', or a false-color blend: see false_color.MODES
HEIGHT_MODE = 'above_ground'  # 'above_ground' or 'elevation'
HEIGHT_CLIP = 10          # color scale max in m; None = 99th percentile of the tiles shown
MAX_YEAR_DIFF = 3           # max |ALS collection year - plot year|
CACHE_DIR = '~/.cache/eagle_als_quiz'

# label -> column. The first TITLE_COVERS are written above each tile; all of them appear in the table below each row.
COVER_COLS = {
    'Shrub': 'AH_ShrubCover', 'Sagebrush': 'AH_SagebrushCover', 'Grass': 'AH_GrassCover', 'Forb': 'AH_ForbCover',
    'Tree': 'FH_TreeCover', 'Litter': 'AH_TotalLitterCover', 'Bare soil': 'BareSoilCover',
    'Invasive': 'AH_InvasiveCover', 'Total foliar': 'TotalFoliarCover', 'Sage':'AH_SagebrushCover'
}
TITLE_COVERS = ['Shrub', 'Tree', 'Sage'] #['Shrub', 'Grass', 'Forb', 'Tree', 'Bare soil']

In [ ]:
import hashlib
import warnings
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from tqdm.auto import tqdm

import false_color as fc
from dataset import load_als_cookie

rng = np.random.default_rng(SEED)
cache_dir = Path(CACHE_DIR).expanduser()
cache_dir.mkdir(parents=True, exist_ok=True)

## Load matched plots

Plots without a matched AWS product, or whose ALS collection is more than `MAX_YEAR_DIFF` years from the field visit, are dropped.

In [ ]:
plots = gpd.read_file(GPKG).drop(columns='geometry')
print(len(plots), 'matched plots in file')
plots = plots.dropna(subset=['product_name_AWS', 'Latitude_WGS84', 'Longitude_WGS84', COVER_COLS.get(SORT_BY, SORT_BY)])
plots = plots[plots['year_diff_AWS'].abs() <= MAX_YEAR_DIFF]
if SPLIT:
    plots = plots[plots['test_split'] == (SPLIT == 'test')]
if STATE:
    plots = plots[plots['State'] == STATE]
plots = plots.reset_index(drop=True)
print(len(plots), 'plots after filters')
plots[list(COVER_COLS.values())].describe().T.round(1)

## Tile loading

`get_tile` streams a cylinder of radius `RADIUS` around the plot center, returns heights relative to the lowest point, and caches the result. Tiles that come back empty or fail to stream are skipped and another plot is drawn.

In [ ]:
def height_above_ground(arr, max_ground=20000):
    """Z minus a ground surface interpolated from ASPRS class-2 points. NaN if there are too few ground points."""
    ground = arr[arr['Classification'] == 2]
    if len(ground) < 50:
        return np.full(len(arr), np.nan, dtype='float32')
    if len(ground) > max_ground:
        ground = ground[np.random.default_rng(0).choice(len(ground), max_ground, replace=False)]
    xy = np.column_stack([ground['X'], ground['Y']])
    q = np.column_stack([arr['X'], arr['Y']])
    surf = LinearNDInterpolator(xy, ground['Z'])(q)
    outside = np.isnan(surf)   # outside the ground-point hull: fall back to nearest ground point
    if outside.any():
        surf[outside] = NearestNDInterpolator(xy, ground['Z'])(q[outside])
    return (arr['Z'] - surf).astype('float32')


def get_tile(row):
    """Return DataFrame[X, Y, Z, HAG, Intensity, ReturnType] (meters, relative to the lowest point / ground) for a plot row, or None if no points."""
    key = hashlib.md5(f"{row['PrimaryKey']}_{RADIUS}_v4".encode()).hexdigest()
    path = cache_dir / f"{key}.parquet"
    if path.exists():
        pts = pd.read_parquet(path)
        return pts if len(pts) >= 100 else None
    try:
        pipeline = load_als_cookie(
            als_tile_name=row['product_name_AWS'], lat=row['Latitude_WGS84'], lon=row['Longitude_WGS84'],
            point_crs='EPSG:4326', radius=RADIUS, out_crs=None,
        )
        arr = pipeline.arrays[0] if pipeline.arrays else []
    except Exception as e:  # network / EPT errors, point outside tile, ...
        warnings.warn(f"{row['PrimaryKey']}: {type(e).__name__}: {str(e)[:100]}")
        return None  # not cached, may work on retry
    if len(arr):
        pts = pd.DataFrame({k: np.asarray(arr[k] - arr[k].min(), dtype='float32') for k in ['X', 'Y', 'Z']})
        pts['Intensity'] = np.asarray(arr['Intensity'], dtype='float32')
        pts['ReturnType'] = fc.return_type_codes(arr['ReturnNumber'], arr['NumberOfReturns'])
        pts['HAG'] = height_above_ground(arr)
    else:
        pts = pd.DataFrame(columns=['X', 'Y', 'Z', 'HAG', 'Intensity', 'ReturnType'], dtype='float32')
    pts.to_parquet(path)
    return pts if len(pts) >= 100 else None


def draw_tiles(candidates, n, exclude=()):
    """Draw n loadable plots from `candidates` (DataFrame), skipping excluded PrimaryKeys. Returns [(row, tile)]."""
    pool = candidates[~candidates['PrimaryKey'].isin(exclude)]
    pool = pool.iloc[rng.permutation(len(pool))]
    out = []
    for _, row in pool.iterrows():
        tile = get_tile(row)
        if tile is not None:
            out.append((row, tile))
        if len(out) == n:
            break
    return out


def heights(tile):
    """Height values to plot (m). Above-ground heights are clipped at 0; tiles with no ground points fall back to raw elevation."""
    if HEIGHT_MODE == 'above_ground' and tile['HAG'].notna().all():
        return tile['HAG'].clip(lower=0)
    return tile['Z']


def point_colors(tile, height_max, color_by):
    """Marker settings for one tile: dict(color, colorscale, cmin, cmax, showscale, colorbar)."""
    if color_by == 'intensity':
        i = tile['Intensity']
        lo, hi = np.percentile(i, [2, 98])   # clip outliers; scale is per tile
        return dict(color=i.clip(lo, hi), colorscale='Cividis', cmin=lo, cmax=hi, showscale=False)
    if color_by in fc.MODES:
        rgb = fc.false_color(heights(tile), tile['Intensity'], tile['ReturnType'], mode=color_by, hag_max=height_max)
        return dict(color=fc.to_plotly_colors(rgb), showscale=False)
    return dict(color=heights(tile), colorscale='Viridis', cmin=0, cmax=height_max, showscale=True,
                colorbar=dict(title='height (m)', len=0.6))


def thin(tile):
    return tile if len(tile) <= MAX_POINTS else tile.iloc[rng.choice(len(tile), MAX_POINTS, replace=False)]


def scene_fig(tiles, titles=None, height_max=None, width=330, color_by=None):
    """One row of 3D scenes, one per tile, sharing a height color scale and axis ranges.
    color_by: one name for all scenes, or a list with one per tile; defaults to COLOR_BY."""
    n = len(tiles)
    color_by = color_by or COLOR_BY
    color_by = [color_by] * n if isinstance(color_by, str) else color_by
    tiles = [thin(t) for t in tiles]
    height_max = height_max or HEIGHT_CLIP or max(np.percentile(heights(t), 99) for t in tiles)
    fig = make_subplots(rows=1, cols=n, specs=[[{'type': 'scene'}] * n], subplot_titles=titles,
                        horizontal_spacing=0.01)
    for i, t in enumerate(tiles, 1):
        fig.add_trace(go.Scatter3d(
            x=t['X'], y=t['Y'], z=heights(t), mode='markers',
            marker=dict(size=1.2, **{**point_colors(t, height_max, color_by[i - 1]),
                                     'showscale': (i == n and color_by[i - 1] == 'height')}),
            hoverinfo='skip',
        ), row=1, col=i)
        # same box for all tiles so scale is comparable: x/y span the tile, z spans the tallest value shown
        fig.update_scenes(dict(
            xaxis=dict(range=[0, 2 * RADIUS], showticklabels=False, title=''),
            yaxis=dict(range=[0, 2 * RADIUS], showticklabels=False, title=''),
            zaxis=dict(range=[0, height_max], title='m'),
            aspectmode='manual', aspectratio=dict(x=1, y=1, z=max(height_max, 2 * RADIUS * 0.4) / (2 * RADIUS)),
        ), row=1, col=i)
    fig.update_layout(width=width * n + 60, height=360, margin=dict(l=0, r=0, t=90, b=0), showlegend=False)
    return fig


## Tiles by cover

One row per quantile bin of `SORT_BY` (lowest cover first). The title of each tile gives the state, visit date and measured cover %. Height color is shared within a row and every tile has the same box size, so structure is comparable. Rotate with the mouse.

In [ ]:
def cover_title(row):
    covers = ' | '.join(f"{k} {row[COVER_COLS[k]]:.0f}%" for k in TITLE_COVERS)
    return f"{row['State']} {row['DateVisited']:%Y-%m-%d}<br>{covers}"


sort_col = COVER_COLS.get(SORT_BY, SORT_BY)
bins = pd.qcut(plots[sort_col].rank(method='first'), N_BINS, labels=False)   # rank first: cover has many ties at 0
shown = {}   # bin -> [(row, tile)]
for b in tqdm(range(N_BINS), desc='bins'):
    shown[b] = draw_tiles(plots[bins == b], N_PER_BIN)
    if len(shown[b]) < N_PER_BIN:
        print(f"bin {b}: only {len(shown[b])} loadable tiles")

In [ ]:
plots['Hgt_Shrub_Avg']

In [ ]:
summary_cols = ['PrimaryKey', 'State', 'CountyName', 'EcolSiteName', 'year_diff_AWS'] + list(COVER_COLS.values()) + ['Hgt_Shrub_Avg']
summary_cols = list(set(summary_cols))

for b, tiles in shown.items():
    lo, hi = plots.loc[bins == b, sort_col].agg(['min', 'max'])
    print(f"\n=== {SORT_BY} bin {b + 1}/{N_BINS}: {lo:.1f} to {hi:.1f}% ({(bins == b).sum()} plots) ===")
    if not tiles:
        continue
    scene_fig([t for _, t in tiles], titles=[cover_title(r) for r, _ in tiles]).show()
    display(pd.DataFrame([r[summary_cols] for r, _ in tiles]).reset_index(drop=True).round(1))

## Look at one plot

Set `KEY` to a `PrimaryKey` from the tables above to view it alone with every cover value, in several color options.

In [ ]:
KEY = None   # e.g. '2023563100103B1'
if KEY:
    row = plots[plots['PrimaryKey'] == KEY].iloc[0]
    tile = thin(get_tile(row))
    display(row[['State', 'CountyName', 'EcolSiteName', 'DateVisited', 'product_name_AWS', 'year_diff_AWS']
                + list(COVER_COLS.values())].to_frame('value'))
    opts = ['height', 'intensity', COLOR_BY]
    scene_fig([tile] * len(opts), titles=opts, color_by=opts).show()